In [1]:
# Raster preparation for Monthly Downscaling Model
# adjust code to consider two variables: tmax and tmin

import os
import gc
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from pathlib import Path

# Configuration
start = pd.Timestamp("2001-01-01")
end = pd.Timestamp("2005-12-31")

dirs = {
    "era5l": r"../Data_Temp/Datasets/ERA5L",
    "har": r"../Data_Temp/Datasets/HAR",
    "chirts": r"../Data_Temp/Datasets/CHIRTS"
}

output_dir = r"../Data_Temp/Processed/Monthly_downscaling/Input_rasters"
os.makedirs(output_dir, exist_ok= True)

template = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif", masked=True) 

In [2]:
# Helpers

def resample_to_template(raster: xr.DataArray) -> xr.DataArray:
    """Reprojects and clips raster to match the 1km WGS84 template"""
    return raster.rio.reproject_match(template, resampling = Resampling.bilinear)

def parse_tif_catalog(directory: str, t_variable: str) -> pd.DataFrame:
    """
    Build a dataframe of (filepath, date) for all tmax and tmin files in the directory
    expected file format {dataset}_{t_variable}_date.tif
    """
    files = sorted(glob.glob(os.path.join(directory, f"*_{t_variable}_*.tif")))
    if not files:
        raise FileNotFoundError(f"No .tif files found for {t_variable} on {directory}")
    rows = []
    for f in files:
        stem = Path(f).stem
        date_str = stem[-8:]
        try:
            date = pd.Timestamp(date_str)
            rows.append({"path": f, "date": date})
        except ValueError:
            continue
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

def filter_period(catalog: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return catalog[(catalog["date"] >= start) & (catalog["date"] >= start)].reset_index(drop=True)

def stack_month(catalog: pd.DataFrame, year: int, month: int) -> xr.DataArray | None:
    """
    Load all daily rasters for year/month, resample each, and stack along
    a new 'day' dimension. Returns None if no files found for that month.
    Memory note: opens one file at a time, closes after stacking.
    """
    month_files = catalog[
        (catalog['date'].dt.year == year) &
        (catalog['date'].dt.month == month)      
    ]
    if month_files.empty:
        return None
    
    slices = []
    for _, row in month_files.iterrows():
        da = rxr.open_rasterio(row["path"], masked = True)
        da = resample_to_template(da)
        if "band" in da.dims and da.sizes["band"] == 1:
            da = da.squeeze("band", drop = True)
        da = da.assign_coords(day = row["date"])
        slices.append(da.load())
        da.close()
    
    stack = xr.concat(slices, dim = "day")

    return stack

def save_raster(data: xr.DataArray, out_path: str, crs, nodata: float = -9999.0):
    """Write a 2D DataArray to GeoTiff, preserving CRS and nodata"""
    data = data.rio.write_crs(crs)
    data = data.rio.write_nodata(nodata)
    data.rio.to_raster(out_path)

def month_label(year: int, month: int) -> str:
    return f"{year}{month:02d}"

# Build a list of (year, month to process)
month_range = pd.period_range(start = start, end = end, freq = "M")

In [3]:
# 1. Temperature products
#  Variables to prepare: Tmax-average, Tmin-average, Tmax-max, Tmin-max, Tmax-min, Tmin-min, Tmax-std, Tmin-std, DTR-Std

temp_products = {
    'era5l': dirs['era5l'],
    'har':dirs['har'],
    'chirts': dirs['chirts']
}

print("Processing temperature products")

for product, directory in temp_products.items():
    product_out = os.path.join(output_dir, product)
    os.makedirs(product_out, exist_ok= True)

    catalog_tmax = filter_period(parse_tif_catalog(directory, "tmax"), start, end)
    catalog_tmin = filter_period(parse_tif_catalog(directory, "tmin"), start, end)
    crs = rxr.open_rasterio(catalog_tmax['path'].iloc[0]).rio.crs

    for period in month_range:
        year, month = period.year, period.month
        label = month_label(year, month)

        #prepare file paths
        tmax_average_path = os.path.join(product_out, f"{product}_tmax_average_{label}.tif")
        tmin_average_path = os.path.join(product_out, f"{product}_tmin_average_{label}.tif")


        extra_exist = True
        if product == "chirts":
            
            tmax_max_path = os.path.join(product_out, f"chirts_tmax_max_{label}.tif")
            tmin_max_path = os.path.join(product_out, f"chirts_tmin_max_{label}.tif")

            tmax_min_path = os.path.join(product_out, f"chirts_tmax_min_{label}.tif")
            tmin_min_path = os.path.join(product_out, f"chirts_tmin_min_{label}.tif")

            tmax_std_path = os.path.join(product_out, f"chirts_tmax_std_{label}.tif")
            tmin_std_path = os.path.join(product_out, f"chirts_tmin_std_{label}.tif")

            dtr_std_path = os.path.join(product_out, f"chirts_dtr_std_{label}.tif")

            extra_exist = (os.path.exists(tmax_max_path) 
                            and os.path.exists(tmin_max_path)
                            and os.path.exists(tmax_min_path)
                            and os.path.exists(tmin_min_path)
                            and os.path.exists(tmax_std_path)
                            and os.path.exists(tmin_std_path)
                            and os.path.exists(dtr_std_path)
            )
        
        if os.path.exists(tmax_average_path) and os.path.exists(tmin_average_path) and extra_exist:
            print(f" [{product}] {label} - skipping (exists)")
            continue

        stack_tmax = stack_month(catalog_tmax, year, month)
        stack_tmin = stack_month(catalog_tmin, year, month)
        
        if stack_tmax is None:
            print(f"No file found for[{product}] Tmax {label}")
        if stack_tmax is None:
            print(f"No file found for[{product}] Tmax {label}")
        
        tmax_monthly_average = stack_tmax.mean(dim = "day", skipna = True)
        tmin_monthly_average = stack_tmin.mean(dim = "day", skipna = True)
        save_raster(tmax_monthly_average, tmax_average_path, crs)
        save_raster(tmin_monthly_average, tmin_average_path, crs)

        #Chirps additional aggregates
        if product == 'chirts':
            tmax_max = stack_tmax.max(dim = "day", skipna = True)
            tmin_max = stack_tmin.max(dim = "day", skipna = True)

            tmax_min = stack_tmax.min(dim = "day", skipna = True)
            tmin_min = stack_tmin.min(dim = "day", skipna = True)

            tmax_std = stack_tmax.std(dim = "day", skipna = True)
            tmin_std = stack_tmin.std(dim = "day", skipna = True)
            dtr_std = (stack_tmax - stack_tmin).std(dim="day", skipna=True)

#            
            save_raster(tmax_max, tmax_max_path, crs)
            save_raster(tmin_max, tmin_max_path, crs)
            save_raster(tmax_min, tmax_min_path, crs)
            save_raster(tmin_min, tmin_min_path, crs)
            save_raster(tmax_std, tmax_std_path, crs)
            save_raster(tmin_std, tmin_std_path, crs)
            save_raster(dtr_std, dtr_std_path, crs)

        del stack_tmax, stack_tmin
        gc.collect()
        print(f" [{product}] {label} - done")
    

Processing temperature products
 [era5l] 200101 - skipping (exists)
 [era5l] 200102 - skipping (exists)
 [era5l] 200103 - skipping (exists)
 [era5l] 200104 - skipping (exists)
 [era5l] 200105 - skipping (exists)
 [era5l] 200106 - skipping (exists)
 [era5l] 200107 - skipping (exists)
 [era5l] 200108 - skipping (exists)
 [era5l] 200109 - skipping (exists)
 [era5l] 200110 - skipping (exists)
 [era5l] 200111 - skipping (exists)
 [era5l] 200112 - skipping (exists)
 [era5l] 200201 - skipping (exists)
 [era5l] 200202 - skipping (exists)
 [era5l] 200203 - skipping (exists)
 [era5l] 200204 - skipping (exists)
 [era5l] 200205 - skipping (exists)
 [era5l] 200206 - skipping (exists)
 [era5l] 200207 - skipping (exists)
 [era5l] 200208 - skipping (exists)
 [era5l] 200209 - skipping (exists)
 [era5l] 200210 - skipping (exists)
 [era5l] 200211 - skipping (exists)
 [era5l] 200212 - skipping (exists)
 [era5l] 200301 - skipping (exists)
 [era5l] 200302 - skipping (exists)
 [era5l] 200303 - skipping (exis

In [4]:
# 2 GLDAS meteorological variables skipped -- can be reused from the precipitation data
# 3. MODIS EVI -- can also be reused
# 4. MODIS LST -- can also be reused